In [1]:
#[FE01] Load preprocessed dataset

import pandas as pd
import numpy as np

df_fe = pd.read_csv(
    "../data/processed/air_quality_cleaned.csv",
    parse_dates=["datetime"]
)

In [2]:
#[FE02] Create next-hour PM2.5 target

df_fe["target_pm25"] = df_fe["pm2.5"].shift(-1)

In [4]:
#[FE03] Check rows with available PM2.5 readings
df_fe.loc[df_fe["pm2.5"].notna(),
          ["datetime", "pm2.5", "target_pm25"]].head(10)

,datetime,pm2.5,target_pm25
24,2010-01-02 00:00:00,129.0,148.0
25,2010-01-02 01:00:00,148.0,159.0
26,2010-01-02 02:00:00,159.0,181.0
27,2010-01-02 03:00:00,181.0,138.0
28,2010-01-02 04:00:00,138.0,109.0
29,2010-01-02 05:00:00,109.0,105.0
30,2010-01-02 06:00:00,105.0,124.0
31,2010-01-02 07:00:00,124.0,120.0
32,2010-01-02 08:00:00,120.0,132.0
33,2010-01-02 09:00:00,132.0,140.0


In [6]:
#[FE04] Create previous-hour PM2.5 feature
df_fe["pm25_lag1"] = df_fe["pm2.5"].shift(1)

In [7]:
#[FE05] Extract day of the week
df_fe["day_of_week"] = df_fe["datetime"].dt.dayofweek

C:\Users\Mkosh\AppData\Local\Temp\ipykernel_22640\1200488371.py:2: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df_fe["day_of_week"] = df_fe["datetime"].dt.dayofweek


In [9]:
#[FE06] Define model features and target

feature_cols = [
    "pm2.5",
    "pm25_lag1",
    "DEWP",
    "TEMP",
    "PRES",
    "Iws",
    "Is",
    "Ir",
    "cbwd_NW",
    "cbwd_SE",
    "cbwd_cv",
    "hour",
    "month",
    "day_of_week"
]

target_col = "target_pm25"

In [12]:
#[FE07] Create modelling dataset and remove missing values

df_model = df_fe[feature_cols + [target_col, "datetime"]].dropna().copy()

df_model = df_model.sort_values("datetime").reset_index(drop=True)

df_model.shape

(41348, 16)

In [13]:
#[FE08] Chronological train-test split

split_index = int(len(df_model) * 0.8)

split_time = df_model.loc[split_index, "datetime"]

train_df = df_model[
    df_model["datetime"].shift(-1) < split_time
].copy()

test_df = df_model[
    df_model["datetime"] >= split_time
].copy()

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

In [14]:
print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))
print("Training period:", train_df["datetime"].min(), "to", train_df["datetime"].max())
print("Testing period:", test_df["datetime"].min(), "to", test_df["datetime"].max())

Training observations: 33077
Testing observations: 8270
Training period: 2010-01-02 01:00:00 to 2014-01-14 16:00:00
Testing period: 2014-01-14 18:00:00 to 2014-12-31 22:00:00


In [15]:
#[FE09] Verify final modelling datasets

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nMissing values in training features:", X_train.isnull().sum().sum())
print("Missing values in testing features:", X_test.isnull().sum().sum())

print("\nFeature columns match:", list(X_train.columns) == list(X_test.columns))

X_train: (33077, 14)
y_train: (33077,)
X_test: (8270, 14)
y_test: (8270,)

Missing values in training features: 0
Missing values in testing features: 0

Feature columns match: True
